In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/itsyashk/profiling-and-reasoning.git'
CLONED_REPO_ROOT = Path('/content/hw2')

if (CLONED_REPO_ROOT / '.git').exists():
    !git -C {CLONED_REPO_ROOT} pull --ff-only
elif CLONED_REPO_ROOT.exists():
    print(f'Using existing non-git repo at {CLONED_REPO_ROOT}')
else:
    !git clone {REPO_URL} {CLONED_REPO_ROOT}


# EE/CS 148B HW2: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime.
- We recommend using an A100 runtime for the GRPO experiments.
- The first cell clones or updates the repo at `/content/hw2`.

This notebook is set up so `Runtime -> Run all` works from a fresh GPU runtime.

Note: We will be using `pip` for dependencies inside Colab.

In [ ]:
%%capture
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" "vllm==0.10.2" "transformers==4.56.1" "datasets>=3.0.0,<4" "accelerate>=1.0,<2" sentencepiece matplotlib pandas tqdm sympy "pylatexenc==2.10" latex2sympy2_extended "math-verify[antlr4-13-2]>=0.7.0" pytest

`vllm` is installed and used by default for the direct, CoT, and self-consistency GSM8K baselines. If vLLM install or initialization fails in Colab, set `USE_VLLM = False` in the config cell to fall back to HuggingFace generation.


In [ ]:
from pathlib import Path

REPO_ROOT = CLONED_REPO_ROOT
assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


In [ ]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

In [ ]:
import gc
import json
import random
import subprocess
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from alignment.eval import load_gsm8k_examples, build_prompts, evaluate_vllm
from alignment.prompts import DIRECT_PROMPT_TEMPLATE, COT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn
from alignment.grpo import train_grpo

SEED = 0
set_seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass


# Your code starts here!

## Config

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-Math-1.5B'
USE_VLLM   = True   # set False to use the slower Transformers fallback
EVAL_N     = None   # None = full GSM8K test split; set 256 only for smoke tests
GRPO_STEPS = 50
K_SC       = 5      # self-consistency K

ARTIFACTS = REPO_ROOT / 'artifacts' / 'colab_results'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

from alignment.eval import (
    load_gsm8k_examples, build_prompts, _extract_ground_truth,
    evaluate_vllm, evaluate_transformers, write_evaluation_results,
    _load_vllm_model, _load_model_and_tokenizer, _make_sampling_params,
)
from alignment.rewards import answer_tag_reward_fn, extract_answer_from_tags
import time

import transformers
try:
    import vllm
    print('vllm =', vllm.__version__)
except Exception as exc:
    print('vllm import failed:', repr(exc))
    USE_VLLM = False
print('torch =', torch.__version__, 'cuda =', torch.version.cuda)
print('transformers =', transformers.__version__)


## Load dataset

In [ ]:
all_test_examples = load_gsm8k_examples('test')
test_examples  = all_test_examples if EVAL_N is None else all_test_examples[:EVAL_N]
train_examples = load_gsm8k_examples('train')
val_examples   = load_gsm8k_examples('test')
ground_truths  = _extract_ground_truth(test_examples)
prompts_direct = build_prompts(test_examples, DIRECT_PROMPT_TEMPLATE)
prompts_cot    = build_prompts(test_examples, COT_PROMPT_TEMPLATE)
print(f'Eval: {len(test_examples)}  Train: {len(train_examples)}')


## §3.1 / §3.2 — Baselines

Skips automatically if results already saved to disk — safe to re-run after restart.

In [ ]:
p31  = ARTIFACTS / 'direct_baseline.json'
pcot = ARTIFACTS / 'cot_baseline.json'
psc  = ARTIFACTS / f'self_consistency_k{K_SC}.json'

def evaluate_with_transformers(prompts, max_new_tokens, num_return_sequences=1):
    model, tokenizer = _load_model_and_tokenizer(MODEL_NAME)
    device = next(model.parameters()).device.type
    out = evaluate_transformers(
        model, tokenizer, answer_tag_reward_fn, prompts, ground_truths,
        max_new_tokens=max_new_tokens, num_return_sequences=num_return_sequences,
        temperature=1.0, device=device,
    )
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    return out

if p31.exists() and pcot.exists() and psc.exists():
    r31    = json.load(open(p31))
    r_cot  = json.load(open(pcot))
    r_sc   = json.load(open(psc))
    print('Loaded baseline results from disk; skipping generation.')
else:
    if USE_VLLM:
        print('Running vLLM baselines ...')
        llm = _load_vllm_model(MODEL_NAME)

        t0  = time.time()
        r31 = evaluate_vllm(llm, answer_tag_reward_fn, prompts_direct,
                            _make_sampling_params(512, n=1),
                            ground_truths=ground_truths, num_return_sequences=1)
        r31['elapsed_seconds'] = time.time() - t0
        write_evaluation_results(r31, p31)

        t0    = time.time()
        r_cot = evaluate_vllm(llm, answer_tag_reward_fn, prompts_cot,
                              _make_sampling_params(1024, n=1),
                              ground_truths=ground_truths, num_return_sequences=1)
        r_cot['elapsed_seconds'] = time.time() - t0
        write_evaluation_results(r_cot, pcot)

        t0   = time.time()
        r_sc = evaluate_vllm(llm, answer_tag_reward_fn, prompts_cot,
                             _make_sampling_params(1024, n=K_SC),
                             ground_truths=ground_truths, num_return_sequences=K_SC)
        r_sc['k'] = K_SC
        r_sc['elapsed_seconds'] = time.time() - t0
        write_evaluation_results(r_sc, psc)

        del llm; gc.collect(); torch.cuda.empty_cache()
        print('\n*** RESTART SESSION NOW before running GRPO (Runtime -> Restart session) ***')
        print('Then re-run from the top; baselines will be skipped automatically.')
    else:
        print('Running slower Transformers baselines ...')
        t0 = time.time(); r31 = evaluate_with_transformers(prompts_direct, 512, 1); r31['elapsed_seconds'] = time.time() - t0; write_evaluation_results(r31, p31)
        t0 = time.time(); r_cot = evaluate_with_transformers(prompts_cot, 1024, 1); r_cot['elapsed_seconds'] = time.time() - t0; write_evaluation_results(r_cot, pcot)
        t0 = time.time(); r_sc = evaluate_with_transformers(prompts_cot, 1024, K_SC); r_sc['k'] = K_SC; r_sc['elapsed_seconds'] = time.time() - t0; write_evaluation_results(r_sc, psc)

for name, r in [('direct', r31), ('CoT', r_cot), (f'SC K={K_SC}', r_sc)]:
    print(f"  {name:10s}: format={r['mean_format_reward']:.3f} answer={r['mean_answer_reward']:.3f} n={r['n']}")


In [ ]:
def category_key(item):
    fmt = int(item.get('format_reward', 0))
    ans = int(item.get('answer_reward', 0))
    return f'format_{fmt}_answer_{ans}'

def summarize_single(results, max_examples=10):
    counts = Counter(category_key(item) for item in results['results'])
    examples = {key: [] for key in counts}
    for item in results['results']:
        key = category_key(item)
        if len(examples[key]) < max_examples:
            examples[key].append(item)
    return {'counts': dict(counts), 'examples': examples}

def summarize_self_consistency(results):
    tie_count = 0
    vote_summaries = []
    for item in results['results']:
        answers = [a for a in (extract_answer_from_tags(r) for r in item['responses']) if a]
        votes = Counter(answers)
        top = votes.most_common()
        is_tie = len(top) > 1 and top[0][1] == top[1][1]
        tie_count += int(is_tie)
        vote_summaries.append({
            'ground_truth': item['ground_truth'],
            'voted_answer': item.get('voted_answer'),
            'votes': dict(votes),
            'tie': is_tie,
            'answer_reward': item['answer_reward'],
        })
    return {
        'tie_count': tie_count,
        'tie_rate': tie_count / max(1, len(results['results'])),
        'vote_summaries': vote_summaries[:50],
    }

baseline_summary = {
    'direct': summarize_single(r31),
    'cot': summarize_single(r_cot),
    'self_consistency': summarize_self_consistency(r_sc),
    'metrics': {
        'direct': {'format': r31['mean_format_reward'], 'answer': r31['mean_answer_reward'], 'n': r31['n']},
        'cot': {'format': r_cot['mean_format_reward'], 'answer': r_cot['mean_answer_reward'], 'n': r_cot['n']},
        'self_consistency': {'format': r_sc['mean_format_reward'], 'answer': r_sc['mean_answer_reward'], 'n': r_sc['n'], 'k': K_SC},
    },
}
json.dump(baseline_summary, open(ARTIFACTS / 'baseline_summary.json', 'w'), indent=2)
print(json.dumps(baseline_summary['metrics'], indent=2))
print('Saved', ARTIFACTS / 'baseline_summary.json')


## §3.5 — GRPO training

**Only run these cells after a fresh session restart** (no vLLM in memory).

In [ ]:
import importlib
import alignment.grpo as _gm
importlib.reload(_gm)
from alignment.grpo import train_grpo
print('train_grpo reloaded from repository source.')


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer_g = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer_g.pad_token_id is None:
    tokenizer_g.pad_token_id = tokenizer_g.eos_token_id

GRPO_KWARGS = dict(
    tokenizer=tokenizer_g,
    train_examples=train_examples, val_examples=val_examples,
    reward_fn=answer_tag_reward_fn,
    prompt_template=str(COT_PROMPT_TEMPLATE),
    n_grpo_steps=GRPO_STEPS, learning_rate=1e-5, advantage_eps=1e-6,
    rollout_batch_size=32, group_size=8, sampling_temperature=1.0,
    sampling_min_tokens=4, sampling_max_tokens=256,
    epochs_per_rollout_batch=1, train_batch_size=32,
    gradient_accumulation_steps=16, cliprange=1.0,
    log_every=1, val_every=5, val_size=256, device=str(DEVICE),
)


In [ ]:
# Run 1: with std normalization
policy_std = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto')
policy_std.train()
t0 = time.time()
hist_std = train_grpo(policy_model=policy_std, normalize_by_std=True, **GRPO_KWARGS)
hist_std['elapsed_seconds'] = time.time() - t0
print(f"Done in {hist_std['elapsed_seconds']/60:.1f} min")
json.dump(hist_std, open(ARTIFACTS / 'grpo_history_std.json', 'w'), indent=2)
del policy_std; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Run 2: without std normalization
policy_nostd = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto')
policy_nostd.train()
t0 = time.time()
hist_nostd = train_grpo(policy_model=policy_nostd, normalize_by_std=False, **GRPO_KWARGS)
hist_nostd['elapsed_seconds'] = time.time() - t0
print(f"Done in {hist_nostd['elapsed_seconds']/60:.1f} min")
json.dump(hist_nostd, open(ARTIFACTS / 'grpo_history_nostd.json', 'w'), indent=2)
del policy_nostd; gc.collect(); torch.cuda.empty_cache()


## Plot + download

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for hist, label in [(hist_std, 'with std norm'), (hist_nostd, 'without std norm')]:
    val_steps = list(range(5, GRPO_STEPS + 1, 5))[:len(hist['val_reward'])]
    ax.plot(val_steps, hist['val_reward'], marker='o', label=label)
ax.set_xlabel('GRPO step'); ax.set_ylabel('Val answer reward')
ax.set_title('GRPO validation reward'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ARTIFACTS / 'grpo_training_curves.png', dpi=150)
plt.show()


In [ ]:
import zipfile
zip_path = REPO_ROOT / 'colab_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in ARTIFACTS.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(REPO_ROOT))
print(f'Zipped: {zip_path}  ({zip_path.stat().st_size/1024**2:.1f} MB)')
from google.colab import files; files.download(str(zip_path))
